# Test one configured OpenAI-compatible model

This notebook verifies that `src/traj_eval/agents/config.py` can load one local configuration and build the expected `LLMConfig`. It uses the model selected by `TRAJ_EVAL_MODEL`, without assuming a provider or model family. The configuration cells are local-only; the explicitly labelled live-probe cells make minimal API requests.

## Instructions

1. Open this worktree's `configs/OpenAi.env` and set `TRAJ_EVAL_MODEL`. Set `OPENAI_BASE_URL` only when using a compatible endpoint rather than OpenAI's default endpoint. Leave `OPENAI_API_KEY` empty to enter it securely in the hidden prompt below.
2. Run the local configuration cells to validate `build_llm_config()`.
3. Select `chat` or `embeddings` for the one configured model.
4. Run **Live availability probe** only when you want to send one minimal request to the configured endpoint.
5. Run **Bulk Traj-Eval chat compatibility probe** only after setting its explicit confirmation flag to `True`.

In [7]:
from pathlib import Path
import os
import sys

START_DIR = Path.cwd().resolve()
ROOT = next((candidate for candidate in (START_DIR, *START_DIR.parents) if (candidate / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError(f'Cannot find the traj-eval repository root from {START_DIR}')
SRC = ROOT / 'src'
src_str = str(SRC)
sys.path[:] = [p for p in sys.path if not p or str(Path(p).resolve()) != src_str]
sys.path.insert(0, src_str)

ENV_PATH = ROOT / 'configs' / 'OpenAi.env'
print('Repository root:', ROOT)
print('Expected env file:', ENV_PATH)
print('Python path starts with repo src:', Path(sys.path[0]).resolve() == SRC)

Repository root: C:\Dev\worktrees\traj-eval\han-lean-anchors-merge
Expected env file: C:\Dev\worktrees\traj-eval\han-lean-anchors-merge\configs\OpenAi.env
Python path starts with repo src: True


In [8]:
print('Loading env file:', ENV_PATH.resolve())

def load_env_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Env file not found: {path}')

    with path.open('r', encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' not in line:
                continue
            key, value = line.split('=', 1)
            key = key.strip()
            value = value.strip()
            if (value.startswith("'") and value.endswith("'")) or (value.startswith('\"') and value.endswith('\"')):
                value = value[1:-1]
            os.environ[key] = value

load_env_file(ENV_PATH)

if not os.environ.get('OPENAI_API_KEY'):
    from getpass import getpass

    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY (stored only in this Jupyter kernel): ')

if not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('An OPENAI_API_KEY is required to build the LLM configuration.')

print('Loaded environment variables from file.')

Loading env file: C:\Dev\worktrees\traj-eval\han-lean-anchors-merge\configs\OpenAi.env
Loaded environment variables from file.


In [9]:
for key in ['OPENAI_BASE_URL', 'OPENAI_API_BASE', 'OPENAI_API_KEY', 'TRAJ_EVAL_MODEL']:
    value = os.environ.get(key)
    if key == 'TRAJ_EVAL_MODEL':
        print(key, repr(value))
    else:
        print(key, 'SET' if value else 'NOT SET')

if os.environ.get('OPENAI_API_BASE') and not os.environ.get('OPENAI_BASE_URL'):
    print('NOTE: config.py uses OPENAI_BASE_URL; OPENAI_API_BASE is not consumed.')

OPENAI_BASE_URL SET
OPENAI_API_BASE SET
OPENAI_API_KEY SET
TRAJ_EVAL_MODEL 'mistral/mistral-large-2512'


In [10]:
from traj_eval.agents.config import build_llm_config

config = build_llm_config()
entry = config.config_list[0]
MODEL_ID = str(entry['model'])
USES_CUSTOM_ENDPOINT = bool(os.environ.get('OPENAI_BASE_URL'))
print('LLMConfig constructed.')
print('Configured model:', MODEL_ID)
print('Custom endpoint configured:', 'YES' if USES_CUSTOM_ENDPOINT else 'NO (OpenAI default)')
print('API key:', 'SET' if entry.get('api_key') else 'MISSING')

LLMConfig constructed.
Configured model: mistral/mistral-large-2512
Custom endpoint configured: YES
API key: SET


## Select the probe type

Test only the configured `TRAJ_EVAL_MODEL`. Use `chat` for a chat-completions model or `embeddings` for an embedding model; this is explicit because model IDs do not reliably describe an endpoint's capability.

In [11]:
PROBE_KIND = 'chat'  # Change to 'embeddings' only for an embedding model.
if PROBE_KIND not in {'chat', 'embeddings'}:
    raise ValueError("PROBE_KIND must be 'chat' or 'embeddings'")

print(f'Selected one {PROBE_KIND} probe for: {MODEL_ID}')

Selected one chat probe for: mistral/mistral-large-2512


## Live availability probe

This cell makes one minimal request to the configured model. It uses the endpoint recorded in `LLMConfig` (or OpenAI's default when no custom base URL is configured) and records only non-secret metadata and latency in `runs/provider_probes/api_probe.csv`. It checks API availability, not Lean or scientific quality.

In [12]:
import csv
from datetime import datetime, timezone
from time import perf_counter

from openai import OpenAI

client_kwargs = {'api_key': str(entry['api_key'])}
base_url = entry.get('base_url')
if base_url:
    client_kwargs['base_url'] = str(base_url)
client = OpenAI(**client_kwargs)

def safe_probe_error(exc: Exception) -> str:
    if type(exc).__name__ == 'AuthenticationError':
        return 'AUTHENTICATION_ERROR: credential rejected by endpoint'
    return f'{type(exc).__name__}: provider details redacted'

started = perf_counter()
try:
    if PROBE_KIND == 'embeddings':
        response = client.embeddings.create(model=MODEL_ID, input=['provider availability probe'])
        response_summary = f'{len(response.data)} embedding(s) returned'
    else:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{'role': 'user', 'content': 'Reply with exactly: OK'}],
            max_tokens=8,
            temperature=0,
        )
        if not response.choices:
            raise RuntimeError('Chat completion contained no choices')
        response_summary = 'chat completion received'
    status = 'WORKS'
    error = ''
except Exception as exc:
    status = 'FAIL'
    response_summary = ''
    error = safe_probe_error(exc)
elapsed = perf_counter() - started

probe_result = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'model': MODEL_ID,
    'probe_kind': PROBE_KIND,
    'custom_base_url': USES_CUSTOM_ENDPOINT,
    'status': status,
    'elapsed_sec': round(elapsed, 3),
    'response_summary': response_summary,
    'error': error,
}

CSV_PATH = ROOT / 'runs' / 'provider_probes' / 'api_probe.csv'
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
with CSV_PATH.open('w', encoding='utf-8', newline='') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=probe_result.keys())
    writer.writeheader()
    writer.writerow(probe_result)

print(f"{MODEL_ID} {status} ({probe_result['elapsed_sec']}s) {error}")
print('CSV:', CSV_PATH)

mistral/mistral-large-2512 WORKS (1.203s) 
CSV: C:\Dev\worktrees\traj-eval\han-lean-anchors-merge\runs\provider_probes\api_probe.csv


## Bulk Traj-Eval chat compatibility probe

This optional cell parses every concrete model ID in `docs/models/models.md`, removes wildcard entries and duplicates, and sends one minimal chat-completions request per model. It uses at most three simultaneous requests and records redacted, timestamped results under the git-ignored `runs/` directory.

The result answers only whether a listed model is compatible with Traj-Eval's chat interface. Image, audio, speech, embedding, moderation, and realtime models can correctly produce `CHAT_ERROR`; that does not mean the model is unavailable or broken. Set `RUN_FULL_CATALOGUE_PROBE` to `True` only when you approve the external requests and their cost.

In [14]:
import csv
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from time import perf_counter

from openai import OpenAI

CATALOGUE_PATH = ROOT / 'docs' / 'models' / 'models.md'
MAX_CONCURRENCY = 3
REQUEST_TIMEOUT_SECONDS = 30
RUN_FULL_CATALOGUE_PROBE = True

if MAX_CONCURRENCY < 1:
    raise ValueError('MAX_CONCURRENCY must be at least 1')
if REQUEST_TIMEOUT_SECONDS <= 0:
    raise ValueError('REQUEST_TIMEOUT_SECONDS must be positive')
if not CATALOGUE_PATH.is_file():
    raise FileNotFoundError(f'Model catalogue not found: {CATALOGUE_PATH}')

model_pattern = re.compile(r'^- `([^`]+)`\s*$')
catalogue_ids = [
    match.group(1)
    for line in CATALOGUE_PATH.read_text(encoding='utf-8').splitlines()
    if (match := model_pattern.match(line))
]
model_ids = list(dict.fromkeys(model_id for model_id in catalogue_ids if not model_id.endswith('/*')))
wildcard_count = len(catalogue_ids) - sum(not model_id.endswith('/*') for model_id in catalogue_ids)

if not model_ids:
    raise RuntimeError(f'No concrete model IDs found in {CATALOGUE_PATH}')

if not RUN_FULL_CATALOGUE_PROBE:
    print(
        f'Ready to probe {len(model_ids)} unique concrete model IDs '
        f'({wildcard_count} wildcard entries excluded). Set RUN_FULL_CATALOGUE_PROBE = True to start.'
    )
else:
    def batch_probe_error(exc: Exception) -> str:
        if type(exc).__name__ == 'AuthenticationError':
            return 'AUTHENTICATION_ERROR: credential rejected by endpoint'
        return f'{type(exc).__name__}: provider details redacted'

    def probe_chat_model(catalogue_index: int, model_id: str) -> dict[str, object]:
        started = perf_counter()
        try:
            model_entry = build_llm_config(model=model_id).config_list[0]
            model_client_kwargs = {'api_key': str(model_entry['api_key'])}
            if model_entry.get('base_url'):
                model_client_kwargs['base_url'] = str(model_entry['base_url'])
            model_client = OpenAI(**model_client_kwargs)

            response = model_client.chat.completions.create(
                model=model_id,
                messages=[{'role': 'user', 'content': 'Reply with exactly: OK'}],
                max_tokens=8,
                temperature=0,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            if not response.choices:
                raise RuntimeError('Chat completion contained no choices')
            status = 'CHAT_COMPATIBLE'
            error = ''
        except Exception as exc:
            status = 'CHAT_ERROR'
            error = batch_probe_error(exc)

        return {
            'catalogue_index': catalogue_index,
            'model': model_id,
            'probe_kind': 'chat.completions',
            'status': status,
            'elapsed_sec': round(perf_counter() - started, 3),
            'error': error,
        }

    batch_started_at = datetime.now(timezone.utc)
    batch_id = batch_started_at.strftime('%Y%m%dT%H%M%S%fZ')
    probe_results = []
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENCY) as executor:
        futures = [
            executor.submit(probe_chat_model, catalogue_index, model_id)
            for catalogue_index, model_id in enumerate(model_ids)
        ]
        for future in as_completed(futures):
            probe_results.append(future.result())

    probe_results.sort(key=lambda result: int(result['catalogue_index']))
    for result in probe_results:
        result['batch_id'] = batch_id
        result['timestamp_utc'] = batch_started_at.isoformat()
        result['catalogue_path'] = str(CATALOGUE_PATH.relative_to(ROOT))
        result['custom_base_url'] = USES_CUSTOM_ENDPOINT

    CSV_PATH = ROOT / 'runs' / 'provider_probes' / f'chat_catalogue_{batch_id}.csv'
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'batch_id', 'timestamp_utc', 'catalogue_path', 'catalogue_index', 'model',
        'probe_kind', 'custom_base_url', 'status', 'elapsed_sec', 'error',
    ]
    with CSV_PATH.open('w', encoding='utf-8', newline='') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(probe_results)

    compatible_count = sum(result['status'] == 'CHAT_COMPATIBLE' for result in probe_results)
    print(f'Chat-compatible: {compatible_count}/{len(probe_results)}')
    print(f'Chat errors: {len(probe_results) - compatible_count}/{len(probe_results)}')
    print('CSV:', CSV_PATH)

Chat-compatible: 74/254
Chat errors: 180/254
CSV: C:\Dev\worktrees\traj-eval\han-lean-anchors-merge\runs\provider_probes\chat_catalogue_20260724T183049437393Z.csv


## One-shot Lean API screening

**Experiment label:** `one-shot model-screening experiment`.

### Goal

Run the same one-shot protocol on any Lean task by changing task data rather than rewriting the workflow. This remains a smoke test, not a benchmark or model ranking. A model succeeds only after independent Lean kernel checking; an API response or plausible proof text is not enough.

### Design pressure and prediction

The sequence is stable: load one task, validate inputs, build one prompt, call every selected model once, and persist auditable artifacts. The task, models, budgets, API caller, and output destination may vary. If those variations stay inside the loop, each new task requires a rewrite. If a new task writes to the same Lean path, it destroys earlier evidence. If an interrupted task is treated as complete, rerunning cannot resume it.

### Smallest useful design

- **Stable orchestrator:** `run_one_shot_lean_screen()` owns the workflow order. It has the fixed-sequence intent associated with Template Method, but ordinary function composition is sufficient here; no inheritance hierarchy is needed.
- **Strategy:** the orchestrator receives `call_strategy` and `persist_strategy` functions, so calling and storage policies can change independently.
- **Adapter:** `final_response_text()` normalizes provider response shapes, while `enrich_one_shot_result()` maps the result into the repository's `TrialMeta`, `TraceEvent`, and `AnchorCheck` vocabulary.
- **State:** a task is `NEW`, `IN_PROGRESS`, `RECOVERABLE`, or `COMPLETE`. Only `COMPLETE` blocks; `IN_PROGRESS` resumes missing model calls from an append-only checkpoint.
- **Task identity and exclusive creation:** a different `task_id` receives its own `model_test_<task_id>.lean`; reusing a completed task ID is rejected instead of overwriting evidence.
- **Protection:** a no-cost validation cell checks the task placeholder, pinned usable models, provider balance, prompt contract, task identity, and output location before any API call.

Classes, factories, builders, commands, and retries are intentionally absent. They add no value for this single stable workflow. The design uses ordinary functions and dictionaries until a real new requirement justifies more structure.

### Evidence boundary

Each model receives the same task source, prompt, temperature, output budget, and timeout exactly once. There are no intentional retries, tools, compiler feedback, or repair turns. After each model attempt, its result is appended immediately to `one_shot_<task_id>.jsonl`. A rerun loads that checkpoint and calls only missing models. Raw responses also go to a final timestamped JSON record, and each distinct task goes to `model_test_<task_id>.lean` for a separate kernel check. The JSON uses trace schema version `0.2.0`; kernel-dependent Lean anchors remain `n/a` until independent validation, so a provider response is never mislabeled as a verified proof. A process killed after an API response but before its checkpoint append creates a small unavoidable uncertainty; that model may be called again because no durable result exists.

In [ ]:
# 1. INPUT INTERFACE — for the next task, edit only this configuration cell.
import json
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from uuid import uuid4

from openai import OpenAI
from traj_eval.trace_core.schema import AnchorCheck, SCHEMA_VERSION, TraceEvent, TrialMeta

TRACE_SCHEMA_VERSION = SCHEMA_VERSION
ONE_SHOT_RECORD_SCHEMA = 'traj-eval.one-shot-lean-screen/1.0.0'

# Use the Lean theorem name as a unique task_id. Keep it unchanged when resuming.
ONE_SHOT_TASK = {
    'task_id': 'leancat_s0001_id_comm',
    'relative_path': Path('dataset/Lean/MiniFATELeanCat/Easy/LeanCat001.lean'),
    'expected_declaration': 'theorem leancat_s0001_id_comm',
    'placeholder': 'sorry',
}
ONE_SHOT_MODELS = [
    'openai/gpt-5.4-2026-03-05',
    'openai/gpt-5.4-mini-2026-03-17',
    'openai/o3-2025-04-16',
    'mistral/magistral-medium-2509',
    'mistral/devstral-2512',
    'mistral/codestral-2508',
]
ONE_SHOT_PROTOCOL = {
    'experiment_label': 'one-shot model-screening experiment',
    'max_tokens': 10240,
    'temperature': 0,
    'timeout_seconds': 120,
    'require_pinned_models': True,
    'require_equal_provider_counts': True,
}
FUTURE_LEAN_CANDIDATE = 'labs-leanstral-1-5'  # Not in the confirmed usable pool; not tested here.
MODEL_SELECTION_PATH = ROOT / 'docs' / 'models' / 'model_selection.md'
ONE_SHOT_OUTPUT_DIR = ROOT / 'runs' / 'provider_probes'

In [ ]:
# 2. Define deterministic steps. These functions make no API calls and write no files.
def load_and_validate_lean_task(root: Path, task_spec: dict[str, object]) -> tuple[Path, str]:
    task_path = (root / Path(task_spec['relative_path'])).resolve()
    if not task_path.is_relative_to(root.resolve()):
        raise ValueError(f'Task must stay inside the repository: {task_path}')
    if not task_path.is_file():
        raise FileNotFoundError(f'Lean task not found: {task_path}')

    task_source = task_path.read_text(encoding='utf-8')
    expected_declaration = str(task_spec['expected_declaration'])
    placeholder = str(task_spec['placeholder'])
    if expected_declaration not in task_source:
        raise ValueError(f'Expected declaration not found: {expected_declaration}')
    if task_source.count(placeholder) != 1:
        raise ValueError(f'Expected exactly one `{placeholder}` placeholder in {task_path}')
    return task_path, task_source

def validate_model_selection(
    model_ids: list[str],
    model_selection_path: Path,
    protocol: dict[str, object],
) -> dict[str, int]:
    duplicate_models = [model_id for model_id, count in Counter(model_ids).items() if count > 1]
    if duplicate_models:
        raise ValueError(f'Duplicate model IDs would violate one call per model: {duplicate_models}')

    usable_pool_text = model_selection_path.read_text(encoding='utf-8')
    missing_models = [model_id for model_id in model_ids if f'- `{model_id}`' not in usable_pool_text]
    if missing_models:
        raise ValueError(f'Models are not recorded in the usable API pool: {missing_models}')

    if protocol['require_pinned_models']:
        mutable_aliases = [model_id for model_id in model_ids if model_id.endswith('-latest')]
        if mutable_aliases:
            raise ValueError(f'Use pinned model IDs for reproducibility: {mutable_aliases}')

    provider_counts = Counter(model_id.split('/', 1)[0] for model_id in model_ids)
    if protocol['require_equal_provider_counts'] and len(set(provider_counts.values())) != 1:
        raise ValueError(f'Provider counts are not balanced: {dict(provider_counts)}')
    return dict(provider_counts)

def build_one_shot_prompt(task_source: str, placeholder: str) -> str:
    return f'''Complete the Lean 4 file below by replacing its single `{placeholder}`.
Return only the replacement proof term beginning with `by`.
Do not return Markdown, the theorem statement, an explanation, `sorry`, or `admit`.
You get one attempt and no compiler feedback.

{task_source}
'''

def safe_task_id(task_id: str) -> str:
    if not re.fullmatch(r'[A-Za-z0-9_-]+', task_id):
        raise ValueError('task_id may contain only letters, digits, underscores, and hyphens')
    return task_id

def task_artifact_paths(output_dir: Path, task_id: str) -> dict[str, Path]:
    safe_id = safe_task_id(task_id)
    return {
        'checkpoint_path': output_dir / f'one_shot_{safe_id}.jsonl',
        'lean_path': output_dir / f'model_test_{safe_id}.lean',
    }

def final_record_paths(output_dir: Path, task_id: str) -> list[Path]:
    safe_id = safe_task_id(task_id)
    return sorted(output_dir.glob(f'one_shot_{safe_id}_????????T????????????Z.json'))

def task_run_state(output_dir: Path, task_id: str) -> dict[str, object]:
    paths = task_artifact_paths(output_dir, task_id)
    records = final_record_paths(output_dir, task_id)
    legacy_path = output_dir / 'model_test.lean'
    legacy_matches = (
        legacy_path.is_file()
        and f'Task: {task_id}' in legacy_path.read_text(encoding='utf-8')
    )
    if paths['lean_path'].exists() or legacy_matches:
        state = 'COMPLETE'
    elif paths['checkpoint_path'].exists():
        state = 'IN_PROGRESS'
    elif records:
        state = 'RECOVERABLE'
    else:
        state = 'NEW'
    return {**paths, 'record_paths': records, 'state': state}

def one_shot_run_contract(
    task_spec: dict[str, object],
    model_ids: list[str],
    protocol: dict[str, object],
) -> dict[str, object]:
    return {
        'task': {
            'task_id': str(task_spec['task_id']),
            'relative_path': str(task_spec['relative_path']).replace('\\', '/'),
            'expected_declaration': str(task_spec['expected_declaration']),
            'placeholder': str(task_spec['placeholder']),
        },
        'models': list(model_ids),
        'protocol': dict(protocol),
    }

def missing_model_ids(model_ids: list[str], completed_results: dict[str, object]) -> list[str]:
    return [model_id for model_id in model_ids if model_id not in completed_results]

def anchor_check(
    name: str,
    status: str,
    *,
    expected: object = None,
    observed: object = None,
    detail: str | None = None,
) -> dict[str, object]:
    if status not in {'pass', 'violation', 'n/a'}:
        raise ValueError(f'Invalid AnchorStatus: {status}')
    return {
        'name': name,
        'status': status,
        'expected': expected,
        'observed': observed,
        'detail': detail,
    }

def build_lean_anchor_checks(result: dict[str, object]) -> list[dict[str, object]]:
    proof_term = str(result.get('proof_term', ''))
    format_status = str(result.get('format_status', ''))
    api_ok = result.get('api_status') == 'API_OK'
    placeholders = sorted(set(re.findall(r'\b(?:sorry|admit)\b', proof_term, re.IGNORECASE)))

    format_anchor = anchor_check(
        'candidate_format',
        'pass' if format_status == 'PROOF_TERM' else ('violation' if api_ok else 'n/a'),
        expected={'starts_with': 'by', 'markdown_fence': False},
        observed={
            'format_status': format_status,
            'starts_with_by': proof_term.startswith('by'),
        },
        detail='API-output contract only; this is not a Lean kernel verdict.',
    )
    placeholder_anchor = anchor_check(
        'prohibited_placeholder_absence',
        ('pass' if not placeholders else 'violation') if proof_term else 'n/a',
        expected=[],
        observed=placeholders if proof_term else None,
        detail='Static check for `sorry` or `admit` in the proposed proof term.',
    )
    pending_kernel_detail = (
        'Pending independent post-hoc Lean validation; no compiler feedback is sent to the model.'
    )
    return [
        format_anchor,
        placeholder_anchor,
        anchor_check(
            'final_proof_compiles',
            'n/a',
            expected={'verification_status': 'accepted', 'compiled': True},
            detail=pending_kernel_detail,
        ),
        anchor_check(
            'final_proof_sorry_free',
            'n/a',
            expected={'sorry_free': True, 'n_sorries': 0},
            detail=pending_kernel_detail,
        ),
        anchor_check(
            'statement_preserved',
            'n/a',
            expected=True,
            detail=pending_kernel_detail,
        ),
        anchor_check(
            'axiom_clean',
            'n/a',
            expected={'axiom_clean': True, 'extra_axioms': []},
            detail=pending_kernel_detail,
        ),
        anchor_check(
            'subgoal_discharge',
            'n/a',
            expected={'fraction': 1.0},
            detail='A final-only one-shot response exposes no tactic-state or subgoal trajectory.',
        ),
    ]

def enrich_one_shot_result(
    result: dict[str, object],
    *,
    task_id: str,
    model_id: str,
    protocol: dict[str, object],
    started_at: str,
    completed_at: str,
) -> dict[str, object]:
    model_token = re.sub(r'[^A-Za-z0-9_-]', '_', model_id).strip('_')
    trial_id = f'{safe_task_id(task_id)}__{model_token}__one_shot'
    anchors = build_lean_anchor_checks(result)
    trial_meta = {
        'trial_id': trial_id,
        'schema_version': TRACE_SCHEMA_VERSION,
        'testbed': 'lean',
        'task_id': task_id,
        'architecture': 'one_shot_api',
        'backbone': model_id,
        'grounding': False,
        'stress_level': 0,
        'started_at': started_at,
        'config': {
            'experiment_label': protocol['experiment_label'],
            'max_tokens': protocol['max_tokens'],
            'temperature': protocol['temperature'],
            'timeout_seconds': protocol['timeout_seconds'],
            'task_source_in_prompt': True,
            'tools_enabled': False,
            'compiler_feedback': False,
            'repair_turns': 0,
        },
    }
    trace_event = {
        'schema_version': TRACE_SCHEMA_VERSION,
        'event_id': str(uuid4()),
        'trial_id': trial_id,
        'seq': 0,
        'timestamp': completed_at,
        'event_type': 'message',
        'agent_role': 'system',
        'caused_by': [],
        'payload': {
            'source': 'one_shot_api_screen',
            'requested_model': model_id,
            'returned_model': result.get('returned_model', ''),
            'response_id': result.get('response_id', ''),
            'api_status': result.get('api_status'),
            'finish_reason': result.get('finish_reason'),
            'format_status': result.get('format_status'),
            'raw_response': result.get('raw_response', ''),
            'proof_term': result.get('proof_term', ''),
        },
        'anchor': anchors[0],
    }
    return {
        **result,
        'schema_version': TRACE_SCHEMA_VERSION,
        'trial_meta': trial_meta,
        'trace_event': trace_event,
        'lean_anchors': anchors,
    }

def build_one_shot_record(
    contract: dict[str, object],
    results: list[dict[str, object]],
) -> dict[str, object]:
    return {
        'schema_version': TRACE_SCHEMA_VERSION,
        'record_schema': ONE_SHOT_RECORD_SCHEMA,
        'schema_references': {
            'trial_meta': 'schema/trial_meta.schema.json',
            'trace_event': 'schema/trace_event.schema.json',
            'anchor_check': 'schema/trace_event.schema.json#/$defs/AnchorCheck',
        },
        'evidence_class': 'provider_probe',
        'scientific_result': False,
        'proposal_mapping': ['RQi', 'O3'],
        'contract': contract,
        'experiment_label': contract['protocol']['experiment_label'],
        'task': contract['task'],
        'protocol': contract['protocol'],
        'models': contract['models'],
        'results': results,
    }

def final_response_text(content: object) -> str:
    if isinstance(content, str):
        return content.strip()
    if not isinstance(content, list):
        return ''

    text_parts = []
    for part in content:
        if isinstance(part, dict) and part.get('type') == 'text':
            text_parts.append(str(part.get('text', '')))
        elif getattr(part, 'type', None) == 'text':
            text_parts.append(str(getattr(part, 'text', '')))
    return '\n'.join(text_parts).strip()

def normalize_proof_term(raw_text: str) -> str:
    text = raw_text.strip()
    fenced = re.fullmatch(r'```(?:lean)?\s*(.*?)\s*```', text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()
    theorem_body = re.search(r':=\s*(by\b.*)', text, flags=re.DOTALL)
    if theorem_body:
        text = theorem_body.group(1).strip()
    return text

def render_lean_candidate_file(
    task_source: str,
    placeholder: str,
    results: list[dict[str, object]],
    experiment_label: str,
) -> str:
    source_lines = task_source.splitlines()
    import_lines = list(dict.fromkeys(
        line.strip() for line in source_lines if line.lstrip().startswith('import ')
    ))
    body = '\n'.join(
        line for line in source_lines if not line.lstrip().startswith('import ')
    ).strip()
    rendered = [f'/- Experiment label: {experiment_label} -/', *import_lines, '']

    for result in results:
        model_id = str(result['requested_model'])
        proof_term = str(result.get('proof_term', ''))
        if result.get('format_status') != 'PROOF_TERM':
            rendered.extend([f'-- {model_id}: no proof term to kernel-check.', ''])
            continue

        namespace = 'Candidate_' + re.sub(r'[^A-Za-z0-9_]', '_', model_id)
        completed_source = body.replace(placeholder, proof_term, 1)
        rendered.extend([
            f'-- Requested model: {model_id}',
            f'namespace {namespace}',
            completed_source,
            f'end {namespace}',
            '',
        ])
    return '\n'.join(rendered).rstrip() + '\n'

In [ ]:
# 3. Define replaceable strategies for the external call and artifact storage.
def write_new_text(path: Path, text: str) -> None:
    # Exclusive creation is the safety boundary: existing evidence is never overwritten.
    with path.open('x', encoding='utf-8', newline='\n') as handle:
        handle.write(text)

def append_checkpoint_event(checkpoint_path: Path, event: dict[str, object]) -> None:
    with checkpoint_path.open('a', encoding='utf-8', newline='\n') as handle:
        handle.write(json.dumps(event, ensure_ascii=False) + '\n')
        handle.flush()

def load_or_start_checkpoint(
    checkpoint_path: Path,
    contract: dict[str, object],
) -> dict[str, dict[str, object]]:
    if not checkpoint_path.exists():
        write_new_text(checkpoint_path, json.dumps({
            'schema_version': TRACE_SCHEMA_VERSION,
            'record_schema': ONE_SHOT_RECORD_SCHEMA,
            'event': 'run_started',
            'recorded_at': datetime.now(timezone.utc).isoformat(),
            'contract': contract,
        }, ensure_ascii=False) + '\n')
        return {}

    events = [
        json.loads(line)
        for line in checkpoint_path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]
    if not events or events[0].get('event') != 'run_started':
        raise ValueError(f'Checkpoint has no run_started event: {checkpoint_path}')
    if events[0].get('contract') != contract:
        raise ValueError(
            'Checkpoint configuration differs from the current input. '
            'Restore the original task/models/protocol or use a new task_id.'
        )

    completed: dict[str, dict[str, object]] = {}
    for event in events[1:]:
        if event.get('event') != 'model_result':
            continue
        result = event.get('result')
        if not isinstance(result, dict):
            raise ValueError(f'Invalid model_result event in {checkpoint_path}')
        model_id = str(result.get('requested_model', ''))
        if model_id not in contract['models']:
            raise ValueError(f'Checkpoint contains an unexpected model: {model_id}')
        if model_id in completed:
            raise ValueError(f'Checkpoint contains duplicate results for: {model_id}')
        completed[model_id] = result
    return completed

def load_recoverable_record(
    record_path: Path,
    contract: dict[str, object],
) -> dict[str, dict[str, object]]:
    record = json.loads(record_path.read_text(encoding='utf-8'))
    if 'contract' in record:
        if record['contract'] != contract:
            raise ValueError(f'Final record does not match current input: {record_path}')
    else:
        if record.get('task', {}).get('task_id') != contract['task']['task_id']:
            raise ValueError(f'Legacy record task does not match current input: {record_path}')
        if record.get('models') != contract['models'] or record.get('protocol') != contract['protocol']:
            raise ValueError(f'Legacy record protocol does not match current input: {record_path}')

    completed = {}
    for result in record.get('results', []):
        model_id = str(result.get('requested_model', ''))
        if model_id in completed:
            raise ValueError(f'Final record contains duplicate results for: {model_id}')
        completed[model_id] = result
    missing = missing_model_ids(list(contract['models']), completed)
    if missing:
        raise ValueError(f'Recoverable final record is missing models: {missing}')
    return completed

def redacted_one_shot_error(exc: Exception) -> str:
    if type(exc).__name__ == 'AuthenticationError':
        return 'AUTHENTICATION_ERROR: credential rejected by endpoint'
    return f'{type(exc).__name__}: provider details redacted'

def call_openai_compatible_one_shot(
    *,
    model_id: str,
    prompt: str,
    task_id: str,
    protocol: dict[str, object],
) -> dict[str, object]:
    started = perf_counter()
    started_at = datetime.now(timezone.utc).isoformat()
    try:
        model_entry = build_llm_config(model=model_id).config_list[0]
        client_kwargs = {'api_key': str(model_entry['api_key'])}
        if model_entry.get('base_url'):
            client_kwargs['base_url'] = str(model_entry['base_url'])
        client = OpenAI(**client_kwargs)
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {'role': 'system', 'content': 'You produce checkable Lean 4 proof terms.'},
                {'role': 'user', 'content': prompt},
            ],
            max_tokens=int(protocol['max_tokens']),
            temperature=float(protocol['temperature']),
            timeout=float(protocol['timeout_seconds']),
        )
        if not response.choices:
            raise RuntimeError('Chat completion contained no choices')

        raw_text = final_response_text(response.choices[0].message.content)
        proof_term = normalize_proof_term(raw_text)
        lowered = proof_term.lower()
        usage = response.usage
        return enrich_one_shot_result({
            'task': task_id,
            'requested_model': model_id,
            'response_id': str(response.id),
            'returned_model': str(response.model),
            'api_status': 'API_OK',
            'elapsed_sec': round(perf_counter() - started, 3),
            'finish_reason': str(response.choices[0].finish_reason),
            'prompt_tokens': getattr(usage, 'prompt_tokens', None),
            'completion_tokens': getattr(usage, 'completion_tokens', None),
            'total_tokens': getattr(usage, 'total_tokens', None),
            'format_status': (
                'PLACEHOLDER_RETURNED' if 'sorry' in lowered or 'admit' in lowered
                else 'PROOF_TERM' if proof_term.startswith('by')
                else 'UNEXPECTED_FORMAT'
            ),
            'raw_response': raw_text,
            'proof_term': proof_term,
            'error': '',
        }, task_id=task_id, model_id=model_id, protocol=protocol,
           started_at=started_at, completed_at=datetime.now(timezone.utc).isoformat())
    except Exception as exc:
        return enrich_one_shot_result({
            'task': task_id,
            'requested_model': model_id,
            'response_id': '',
            'returned_model': '',
            'api_status': 'API_ERROR',
            'elapsed_sec': round(perf_counter() - started, 3),
            'finish_reason': '',
            'prompt_tokens': None,
            'completion_tokens': None,
            'total_tokens': None,
            'format_status': 'NO_PROOF',
            'raw_response': '',
            'proof_term': '',
            'error': redacted_one_shot_error(exc),
        }, task_id=task_id, model_id=model_id, protocol=protocol,
           started_at=started_at, completed_at=datetime.now(timezone.utc).isoformat())

def persist_one_shot_artifacts(
    *,
    output_dir: Path,
    task_spec: dict[str, object],
    task_source: str,
    protocol: dict[str, object],
    contract: dict[str, object],
    results: list[dict[str, object]],
    existing_record_path: Path | None = None,
) -> dict[str, object]:
    output_dir.mkdir(parents=True, exist_ok=True)
    run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    task_id = str(task_spec['task_id'])
    safe_id = safe_task_id(task_id)
    record_path = existing_record_path or output_dir / f'one_shot_{safe_id}_{run_stamp}.json'
    lean_path = task_artifact_paths(output_dir, task_id)['lean_path']
    if lean_path.exists():
        raise FileExistsError(f'Completed Lean artifact already exists: {lean_path}')
    record = build_one_shot_record(contract, results)
    if existing_record_path is None:
        write_new_text(record_path, json.dumps(record, indent=2, ensure_ascii=False) + '\n')
    else:
        existing_record = json.loads(record_path.read_text(encoding='utf-8'))
        if existing_record.get('results') != results:
            raise ValueError(f'Existing final record disagrees with checkpoint: {record_path}')
    write_new_text(
        lean_path,
        render_lean_candidate_file(
            task_source,
            str(task_spec['placeholder']),
            results,
            str(protocol['experiment_label']),
        ),
    )
    return {'record_path': record_path, 'lean_path': lean_path}

In [ ]:
# 4. Protection checks: run this cell before spending API budget.
validated_task_path, validated_task_source = load_and_validate_lean_task(ROOT, ONE_SHOT_TASK)
validated_provider_counts = validate_model_selection(
    ONE_SHOT_MODELS,
    MODEL_SELECTION_PATH,
    ONE_SHOT_PROTOCOL,
)
validated_prompt = build_one_shot_prompt(validated_task_source, str(ONE_SHOT_TASK['placeholder']))
validated_state = task_run_state(ONE_SHOT_OUTPUT_DIR, str(ONE_SHOT_TASK['task_id']))
if validated_state['state'] == 'COMPLETE':
    raise FileExistsError(
        f"Task `{ONE_SHOT_TASK['task_id']}` is COMPLETE; choose a new task_id."
    )
synthetic_candidate = render_lean_candidate_file(
    'import Mathlib\n\ntheorem transfer_example : True := __HOLE__\n',
    '__HOLE__',
    [{
        'requested_model': 'test/model-1',
        'format_status': 'PROOF_TERM',
        'proof_term': 'by trivial',
    }],
    'protection check',
)
synthetic_result = enrich_one_shot_result(
    {
        'task': 'transfer_example',
        'requested_model': 'test/model-1',
        'response_id': 'synthetic-response',
        'returned_model': 'test/model-1',
        'api_status': 'API_OK',
        'finish_reason': 'stop',
        'format_status': 'PROOF_TERM',
        'raw_response': 'by trivial',
        'proof_term': 'by trivial',
    },
    task_id='transfer_example',
    model_id='test/model-1',
    protocol=ONE_SHOT_PROTOCOL,
    started_at='2026-07-25T00:00:00+00:00',
    completed_at='2026-07-25T00:00:01+00:00',
)
synthetic_contract = one_shot_run_contract(
    {
        'task_id': 'transfer_example',
        'relative_path': Path('dataset/Lean/synthetic.lean'),
        'expected_declaration': 'theorem transfer_example',
        'placeholder': '__HOLE__',
    },
    ['test/model-1'],
    ONE_SHOT_PROTOCOL,
)
synthetic_record = build_one_shot_record(synthetic_contract, [synthetic_result])
synthetic_anchors = {anchor['name']: anchor for anchor in synthetic_result['lean_anchors']}
TrialMeta.model_validate(synthetic_result['trial_meta'])
TraceEvent.model_validate(synthetic_result['trace_event'])
for synthetic_anchor in synthetic_result['lean_anchors']:
    AnchorCheck.model_validate(synthetic_anchor)

assert normalize_proof_term('```lean\nby\n  exact h\n```') == 'by\n  exact h'
assert str(ONE_SHOT_TASK['placeholder']) in validated_prompt
assert FUTURE_LEAN_CANDIDATE not in ONE_SHOT_MODELS
assert ONE_SHOT_OUTPUT_DIR.is_relative_to(ROOT)
assert task_artifact_paths(ONE_SHOT_OUTPUT_DIR, 'task_a') != task_artifact_paths(ONE_SHOT_OUTPUT_DIR, 'task_b')
assert missing_model_ids(['a', 'b', 'c'], {'a': {}}) == ['b', 'c']
assert '__HOLE__' not in synthetic_candidate and 'by trivial' in synthetic_candidate
assert synthetic_result['trial_meta']['testbed'] == 'lean'
assert synthetic_result['trace_event']['agent_role'] == 'system'
assert synthetic_result['trace_event']['anchor'] == synthetic_anchors['candidate_format']
assert synthetic_anchors['candidate_format']['status'] == 'pass'
assert synthetic_anchors['final_proof_compiles']['status'] == 'n/a'
assert synthetic_record['schema_version'] == SCHEMA_VERSION
assert synthetic_record['schema_references']['anchor_check'].endswith('#/$defs/AnchorCheck')
assert synthetic_record['scientific_result'] is False

print('Task validated:', ONE_SHOT_TASK['task_id'])
print('Task source:', validated_task_path.relative_to(ROOT))
print('Provider balance:', validated_provider_counts)
print('Models:', len(ONE_SHOT_MODELS))
print('Task state:', validated_state['state'])
print('Output directory:', ONE_SHOT_OUTPUT_DIR.relative_to(ROOT))
print('Checkpoint:', validated_state['checkpoint_path'].name)
print('Lean artifact:', validated_state['lean_path'].name)
print('Future candidate excluded:', FUTURE_LEAN_CANDIDATE)

In [ ]:
# 5. Stable workflow template. Variation enters only through arguments.
def run_one_shot_lean_screen(
    *,
    root: Path,
    task_spec: dict[str, object],
    model_ids: list[str],
    protocol: dict[str, object],
    model_selection_path: Path,
    output_dir: Path,
    call_strategy=call_openai_compatible_one_shot,
    persist_strategy=persist_one_shot_artifacts,
) -> dict[str, object]:
    task_path, task_source = load_and_validate_lean_task(root, task_spec)
    provider_counts = validate_model_selection(model_ids, model_selection_path, protocol)
    output_dir.mkdir(parents=True, exist_ok=True)
    task_id = str(task_spec['task_id'])
    state_info = task_run_state(output_dir, task_id)
    if state_info['state'] == 'COMPLETE':
        raise FileExistsError(f'Task `{task_id}` is COMPLETE; use a new task_id.')

    contract = one_shot_run_contract(task_spec, model_ids, protocol)
    prompt = build_one_shot_prompt(task_source, str(task_spec['placeholder']))
    checkpoint_path = state_info['checkpoint_path']
    record_paths = list(state_info['record_paths'])
    if len(record_paths) > 1:
        raise ValueError(f'Multiple final records make resume ambiguous: {record_paths}')
    existing_record_path = record_paths[0] if record_paths else None

    if state_info['state'] == 'RECOVERABLE':
        completed_results = load_recoverable_record(existing_record_path, contract)
    else:
        completed_results = load_or_start_checkpoint(checkpoint_path, contract)

    resumed_model_ids = [model_id for model_id in model_ids if model_id in completed_results]
    called_model_ids = []
    # Sequential calls plus immediate append make interruption recovery explicit.
    for model_id in missing_model_ids(model_ids, completed_results):
        result = call_strategy(
            model_id=model_id,
            prompt=prompt,
            task_id=task_id,
            protocol=protocol,
        )
        append_checkpoint_event(checkpoint_path, {
            'schema_version': TRACE_SCHEMA_VERSION,
            'event': 'model_result',
            'recorded_at': datetime.now(timezone.utc).isoformat(),
            'trial_id': result['trial_meta']['trial_id'],
            'result': result,
        })
        completed_results[model_id] = result
        called_model_ids.append(model_id)

    results = [completed_results[model_id] for model_id in model_ids]
    if existing_record_path is not None and state_info['state'] == 'IN_PROGRESS':
        recovered_results = load_recoverable_record(existing_record_path, contract)
        if [recovered_results[model_id] for model_id in model_ids] != results:
            raise ValueError('Final record disagrees with the append-only checkpoint')

    artifacts = persist_strategy(
        output_dir=output_dir,
        task_spec=task_spec,
        task_source=task_source,
        protocol=protocol,
        contract=contract,
        results=results,
        existing_record_path=existing_record_path,
    )
    if checkpoint_path.exists():
        append_checkpoint_event(checkpoint_path, {
            'schema_version': TRACE_SCHEMA_VERSION,
            'event': 'finalized',
            'recorded_at': datetime.now(timezone.utc).isoformat(),
            'record_path': artifacts['record_path'].name,
            'lean_path': artifacts['lean_path'].name,
        })
    return {
        'task_path': task_path,
        'state_before_run': state_info['state'],
        'provider_counts': provider_counts,
        'checkpoint_path': checkpoint_path,
        'resumed_model_ids': resumed_model_ids,
        'called_model_ids': called_model_ids,
        'results': results,
        **artifacts,
    }

RUN_ONE_SHOT_LEAN_SCREEN = False

preview_state = task_run_state(ONE_SHOT_OUTPUT_DIR, str(ONE_SHOT_TASK['task_id']))
if preview_state['state'] == 'IN_PROGRESS':
    preview_contract = one_shot_run_contract(ONE_SHOT_TASK, ONE_SHOT_MODELS, ONE_SHOT_PROTOCOL)
    preview_completed = load_or_start_checkpoint(preview_state['checkpoint_path'], preview_contract)
    preview_remaining = missing_model_ids(ONE_SHOT_MODELS, preview_completed)
elif preview_state['state'] == 'NEW':
    preview_remaining = list(ONE_SHOT_MODELS)
else:
    preview_remaining = []

if not RUN_ONE_SHOT_LEAN_SCREEN:
    print(
        f"Task state: {preview_state['state']}. "
        f'{len(preview_remaining)} model calls remain. '
        'Set RUN_ONE_SHOT_LEAN_SCREEN = True only after the protection checks pass.'
    )
else:
    one_shot_run = run_one_shot_lean_screen(
        root=ROOT,
        task_spec=ONE_SHOT_TASK,
        model_ids=ONE_SHOT_MODELS,
        protocol=ONE_SHOT_PROTOCOL,
        model_selection_path=MODEL_SELECTION_PATH,
        output_dir=ONE_SHOT_OUTPUT_DIR,
    )
    print('JSON record:', one_shot_run['record_path'])
    print('Lean candidates:', one_shot_run['lean_path'])
    print('Resumed models:', one_shot_run['resumed_model_ids'])
    print('Called now:', one_shot_run['called_model_ids'])
    print('Kernel command: lake env lean', one_shot_run['lean_path'].relative_to(ROOT))
    for result in one_shot_run['results']:
        print(result['requested_model'], result['api_status'], result['format_status'])

### Transfer: run a different Lean task

1. For a new task, change only `ONE_SHOT_TASK`: use its unique theorem name as `task_id`, plus its repository-relative `.lean` path, declaration prefix, and placeholder. To resume an interrupted task, keep this input unchanged.
2. Keep `ONE_SHOT_MODELS` unchanged for a matched task comparison, or replace it only with pinned IDs already present in `docs/models/model_selection.md`.
3. Run the protection cell. It must pass before any paid API call.
4. Set `RUN_ONE_SHOT_LEAN_SCREEN = True` and run the final cell. If execution is interrupted, rerun the same cell with the same input; recorded models are skipped and only missing model calls continue.
5. Run the printed `lake env lean ...` command from the repository root. Do not feed errors back to models in a one-shot experiment.
6. Interpret JSON/API status and Lean kernel status separately. The final JSON embeds schema-validated `TrialMeta`, `TraceEvent`, and Lean `AnchorCheck` records; kernel-dependent anchors stay `n/a` until the independent check. The task receives an append-only checkpoint and its own `model_test_<task_id>.lean`; prior task files are not changed.

### Why this pattern, and when not to use it

- Compared with **no pattern**, task-specific data no longer leaks into the API loop or output naming, and later tasks cannot silently overwrite earlier evidence.
- Compared with a class-based **Template Method**, the orchestrator plus function strategies preserve the stable sequence with less ceremony and no inheritance coupling.
- **State** is now justified because `NEW`, `IN_PROGRESS`, `RECOVERABLE`, and `COMPLETE` require different behavior. A single existence check caused the resume bug.
- Compared with a **Factory**, there is no repeated object-construction decision yet; adding one would be speculative.
- Compared with **Command**, the checkpoint records outcomes but does not represent queued or undoable requests. State plus an append-only event log is enough.

A useful transfer exercise is to select one Medium LeanCat file containing exactly one `sorry`, assign its theorem/task name as the new `task_id`, and change only `ONE_SHOT_TASK`. Interrupt a dry-run strategy after one result, rerun with the same input, and confirm that the recorded model is skipped. Reusing a `COMPLETE` task such as `leancat_s0001_id_comm` remains blocked.

## Notes

- `build_llm_config()` only verifies environment loading and config construction; it does not call an API by itself.
- The **Live availability probe**, **Bulk Traj-Eval chat compatibility probe**, and guarded **One-shot Lean API screening** cells make API requests.
- If the env file has no key, the setup cell prompts for it without displaying or saving it; the key exists only in the active Jupyter kernel.
- The live probe tests endpoint availability only; it is not a Lean or scientific evaluation.
- The single-model probe does not rely on model listing, model-name prefixes, or a provider-specific model catalogue.
- The bulk probe reads `docs/models/models.md`, excludes wildcards and duplicates, and tests chat compatibility only.
- All probes write non-secret results to the git-ignored `runs/` directory; successful probes are operational evidence, not scientific results.